# 40 — Predictive Analytics & Comparative Survival Workbench (Phase 4.5)

Demonstrates the `ThyroidPredictiveAnalyzer` from `utils/predictive_analytics.py`:

1. **PTCM-powered cure prediction** — individual patient scoring + sensitivity analysis
2. **Competing risks** — Aalen-Johansen CIF with Gray's landmark tests + KM overlay
3. **Multi-model comparison** — 6 models: KM, Cox PH, PTCM, Mixture Cure, Penalized Cox, RSF
4. **ML nomograms** — XGBoost + SHAP with force plots
5. **Manuscript report** — one-click Word doc generation

**Phase 4.5 enhancements (v2026.03.11):**
- Gray's landmark CIF equality tests (Pepe-Mori z-test approximation)
- KM overlay and CI bands on CIF curves
- Sensitivity analysis (what-if scenarios) for cure calculator
- Advanced clinical context (NSQIP complications, RAI status)
- Model comparison recommendation engine

**Requires:** MotherDuck token (or local DuckDB with materialized tables).

```bash
export MOTHERDUCK_TOKEN='your_token'
jupyter notebook notebooks/40_predictive_analytics_examples.ipynb
```

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

import duckdb
import pandas as pd

# Connect to MotherDuck (or local)
USE_LOCAL = os.environ.get("USE_LOCAL_DUCKDB", "false").lower() == "true"

if USE_LOCAL:
    con = duckdb.connect(str(ROOT / "thyroid_master.duckdb"))
    print("Connected to local DuckDB")
else:
    token = os.environ.get("MOTHERDUCK_TOKEN", "")
    if not token:
        import toml
        token = toml.load(str(ROOT / ".streamlit" / "secrets.toml")).get("MOTHERDUCK_TOKEN", "")
    con = duckdb.connect(f"md:?motherduck_token={token}")
    con.execute("USE thyroid_research_2026;")
    print("Connected to MotherDuck")

In [ ]:
from utils.predictive_analytics import (
    ThyroidPredictiveAnalyzer,
    PREDICTIVE_PRESETS,
    CURE_CALCULATOR_FEATURES,
    CLINICAL_INTERPRETATIONS,
)

pa = ThyroidPredictiveAnalyzer(con)
print(f"PTCM available: {pa.ptcm_available}")
print(f"Calculator features: {list(CURE_CALCULATOR_FEATURES.keys())}")
print(f"Presets: {list(PREDICTIVE_PRESETS.keys())}")

## 1. Individual Cure Prediction (PTCM)

Score a single patient with the Weibull PTCM, including hybrid adjustment
for tumor size and lymph node status.

In [ ]:
# Example: 52-year-old with Stage II, microscopic ETE, BRAF+, 2.3cm tumor, N1a
patient = {
    "age_at_diagnosis": 52,
    "ajcc_stage_8": "II",
    "ete_type": "microscopic",
    "braf_status": True,
    "tert_status": False,
    "recurrence_risk_band": "intermediate",
    "tumor_size_cm": 2.3,
    "ln_status": "N1a",
}

result = pa.predict_individual_cure_probability(patient)

if "error" not in result:
    print(f"Cure probability: {result['cure_probability_pct']}%")
    print(f"Tier: {result['cure_tier']}")
    print(f"theta: {result['theta']}")
    print(f"\nClinical interpretation:\n{result['cure_interpretation']}")
    print(f"\nConditional survival:")
    display(result["conditional_survival"])
    if result.get("warnings"):
        print(f"\nWarnings: {result['warnings']}")
else:
    print(f"Error: {result['error']}")

In [ ]:
# Personalized survival trajectory plot
fig = pa.plot_individual_cure_trajectory(patient)
if fig:
    fig.show()

In [ ]:
# Sensitivity analysis — which features matter most for this patient?
sens = pa.sensitivity_analysis(patient)
if not sens.empty:
    print("Sensitivity analysis (sorted by absolute cure-probability swing):")
    display(sens)
else:
    print("Sensitivity analysis unavailable.")

In [ ]:
# Advanced clinical context — NSQIP complications + RAI
# These don't change the PTCM theta but modify clinical interpretation
patient_nsqip = {
    **patient,
    "any_nsqip_complication": True,
    "rln_injury": False,
    "hypocalcemia": True,
    "rai_received": True,
}
result_adv = pa.predict_individual_cure_probability(patient_nsqip)
if "error" not in result_adv:
    print(f"Cure probability (unchanged by NSQIP): {result_adv['cure_probability_pct']}%")
    if result_adv.get("advanced_clinical_note"):
        print(f"\nAdvanced clinical context:\n{result_adv['advanced_clinical_note']}")
else:
    print(result_adv["error"])

## 2. Competing Risks Analysis

Aalen-Johansen CIF with cause-specific Cox HRs.

In [ ]:
preset = PREDICTIVE_PRESETS["recurrence"]
cr = pa.fit_competing_risks(
    time_col=preset["time_col"],
    event_col=preset["event_col"],
    competing_event_col=preset["competing_event_col"],
    predictors=preset["predictors"],
    strata_col="overall_stage_ajcc8",
)

if "error" not in cr:
    print(f"N={cr['n_obs']:,} | Primary events={cr['n_events']} | Competing={cr['n_competing']}")

    # Landmark CIF summary
    print("\n--- Landmark CIF Estimates ---")
    display(cr["summary_table"])

    # CIF plot with KM overlay and CI bands
    if cr.get("cif_plot"):
        cr["cif_plot"].show()

    # Gray's landmark CIF equality tests (Phase 4.5)
    gray = cr.get("gray_tests")
    if gray is not None and not gray.empty:
        print("\n--- Gray's Landmark CIF Tests (Pepe-Mori z-test) ---")
        display(gray)

    # Cause-specific log-rank tests
    cs_lr = cr.get("cause_specific_logrank", {})
    for label, lr_df in cs_lr.items():
        print(f"\n--- Cause-Specific Log-Rank ({label}) ---")
        display(lr_df)

    # Cause-specific Cox HRs
    for label, hr in cr.get("cause_specific_hrs", {}).items():
        print(f"\n--- Cause-Specific Cox ({label}) — C-index: {hr['concordance']} ---")
        display(hr["hr_table"])

    print(f"\n{cr.get('clinical_note', '')}")
else:
    print(cr["error"])

## 3. Multi-Model Comparison

KM, Cox PH, PTCM, Mixture Cure, Penalized Cox, Random Survival Forest.

In [ ]:
comp = pa.compare_survival_models()

if "error" not in comp:
    print(f"Models compared: {comp['n_models']}")

    # Clinical recommendation (Phase 4.5)
    rec = comp.get("recommendation", "")
    if rec:
        print(f"\n--- Recommendation ---\n{rec}\n")

    display(comp["comparison_table"])
    if comp.get("comparison_plot"):
        comp["comparison_plot"].show()
    if comp.get("warnings"):
        for w in comp["warnings"]:
            print(f"  WARN: {w}")
else:
    print(comp["error"])

## 4. ML Nomogram + SHAP

XGBoost for structural recurrence with SHAP explainability.

In [ ]:
nom = pa.train_explainable_nomogram(
    outcome="event_occurred",
    predictors=preset["predictors"][:8],
    base_model="xgboost",
)

if "error" not in nom:
    print(f"AUC: {nom['auc_cv']} | Brier: {nom['brier_cv']} | N: {nom['n_obs']:,}")
    if nom.get("shap_summary_plot"):
        nom["shap_summary_plot"].show()
    if nom.get("shap_beeswarm_plot"):
        nom["shap_beeswarm_plot"].show()
    if nom.get("calibration_plot"):
        nom["calibration_plot"].show()
    display(nom["feature_importance"])
else:
    print(nom["error"])

## 5. Manuscript Report Generation

One-click Word document with all analysis sections.

In [ ]:
report = pa.generate_manuscript_report(
    sections=["PTCM", "CompetingRisks", "Comparison"],
    title="THYROID_2026 — Predictive Analytics Report",
    author="Thyroid Research Team",
)

if "error" not in report:
    print(f"Report saved: {report['path']}")
    print(f"Sections: {report['sections_included']}")
else:
    print(report["error"])

In [ ]:
con.close()
print("Done.")